Following the data and simulation files processing, here we:

- Merge processed files by production conditions.
- Tag global events by type of particle (electron or alpha-like) and detector region.
- Store final HDF5 merged file for subsequent analysis.

In [1]:
import sys
sys.path.append('/lhome/ific/c/ccortesp/Analysis/')

from libs import crudo

from datetime import datetime
import glob
import os
import pandas as pd
import numpy as np

%matplotlib inline
%load_ext autoreload
%autoreload 2

# Configuration

In [2]:
# ----- Notebook Details
TYPE = 'mc'                     # Options: 'data', 'mc'
FILE_TAG = 'calibration_lpr'    # Options: 'radiogenics_hpr', 'radiogenics_lpr', 'bb2nu_hpr', 'bb2nu_lpr', 'bb0nu_hpr', 'bb0nu_lpr', 'p2_zemrude', 'p2_icaros', 'p2_final'

# Data
DATA_PERIOD = 2                 # Options: 1, 2
DETECTOR_CONDITION = 'castle_closed_RAS'       # Options: None, 'castle_open', 'castle_closed', 'castle_closed_RAS', 'castle_pclosed', 'castle_pclosed_RAS'

# MC
DATE = datetime.now().strftime('%d%m%Y')    # Options: today or some day (e.g '02122025')
# DATE = '12052026'

In [3]:
# ------------------------------
# DIRECTORIES, PATHS & FILENAMES
# ------------------------------
PROC_DATA_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/runs/'
PROC_MC_DIR   = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/mc/'
OUTPUT_DIR = '/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/'

RUNS_INFO_PATH = os.path.join('/lhome/ific/c/ccortesp/Analysis/NEXT-100/Backgrounds/utilities/runs_information.csv')

SUMMARY_FILENAME = 'summary_' + FILE_TAG +'_processed.csv'
SUMMARY_PATH = os.path.join('/lhome/ific/c/ccortesp/Analysis/NEXT-100/Backgrounds/txt/summaries/', SUMMARY_FILENAME)

# -----------------------------
# ALPHA/ELECTRON DISCRIMINATION
# -----------------------------
SIZE_THRESHOLD = 2e3          # in [# of hits]
S1_ENERGY_THRESHOLD = 900     # in [PE]

# ----------------
# DETECTOR REGIONS
# ----------------
# Geometric boundaries for event classification.
Z_LOW = 40          # in [mm]
Z_UP  = 1147        # in [mm]
R_UP  = 451.65      # in [mm]

# -------------
# FINAL COLUMNS
# -------------
INFO_COLS = ['event', 'global_event', 'time']
MC_COLS   = ['isotope', 'volume', 'double_e', 'region']
DATA_COLS = ['run_number', 'particle', 'region']
EVT_COLS = ['nS1', 'nS2', 'n_cluster', 'old_n_hits', 'n_hits', 'E_evt' if TYPE == 'mc' else 'E_evt_pe']      # CHANGE TO MEV SOON
POS_COLS = ['X_bary', 'Y_bary', 'Z_bary', 'X_min', 'X_max', 'Y_min', 'Y_max', 'Z_min', 'Z_max', 'R_max']
S1_PULSE_COLS = ['S1e', 'S1e_corr', 'S1w', 'S1h', 'S1t']
S2_PULSE_COLS = ['S2e', 'S2w', 'S2h', 'S2t', 'S2q']

if TYPE == 'mc':
    FINAL_COLS = INFO_COLS + MC_COLS + EVT_COLS + POS_COLS + S1_PULSE_COLS + S2_PULSE_COLS
elif TYPE == 'data':
    FINAL_COLS = INFO_COLS + DATA_COLS + EVT_COLS + POS_COLS + S1_PULSE_COLS + S2_PULSE_COLS  

In [4]:
FINAL_COLS

['event',
 'global_event',
 'time',
 'isotope',
 'volume',
 'double_e',
 'region',
 'nS1',
 'nS2',
 'n_cluster',
 'old_n_hits',
 'n_hits',
 'E_evt_mev',
 'X_bary',
 'Y_bary',
 'Z_bary',
 'X_min',
 'X_max',
 'Y_min',
 'Y_max',
 'Z_min',
 'Z_max',
 'R_max',
 'S1e',
 'S1e_corr',
 'S1w',
 'S1h',
 'S1t',
 'S2e',
 'S2w',
 'S2h',
 'S2t',
 'S2q']

### Runs & Summary Information

In [5]:
# Runs information
RUNS_INFO_DF = pd.read_csv(RUNS_INFO_PATH)
RUNS_INFO_DF.columns = RUNS_INFO_DF.columns.str.strip()
RUNS_INFO_DF

,run_number,duration,OK,LOST,period,condition
0,15062,84783,69564,1339,1,castle_open
1,15063,79120,65052,1241,1,castle_open
2,15076,69316,56775,1080,1,castle_open
3,15288,87256,30201,8397,1,castle_pclosed_RAS
4,15289,82152,28180,7884,1,castle_pclosed_RAS
...,...,...,...,...,...,...
106,15733,86919,30475,10027,2,castle_closed_RAS
107,15734,85790,29837,9598,2,castle_closed_RAS
108,15735,87451,30547,9958,2,castle_closed_RAS
109,15736,93376,32622,10506,2,castle_closed_RAS


In [4]:
# Summary of the processed runs
SUMMARY_DF = pd.read_csv(SUMMARY_PATH)
SUMMARY_DF.drop(columns=['Unnamed: 0'], inplace=True)
SUMMARY_DF.columns = SUMMARY_DF.columns.str.strip()
if TYPE == 'data':  SUMMARY_DF.sort_values(by='Run_ID', inplace=True)
SUMMARY_DF

,Isotope,Generated,Interacting,Saved,Sophronia,Clean,Strong_S2,S1_Cut
0,Tl208,5000000,901498,132594,131397,131384,131384,131001


# Merge by Production Condition

- For data, the production are differenciated by _data period_ and _detector condition_.
- For MC, by type of simulation.

### Data

In [9]:
# Select runs to use according to the notebook configuration
if DATA_PERIOD is not None:
    runs_to_analyze = RUNS_INFO_DF.loc[RUNS_INFO_DF['period'] == DATA_PERIOD, 'run_number'].values
    if DETECTOR_CONDITION is not None:
        runs_to_analyze = RUNS_INFO_DF.loc[(RUNS_INFO_DF['period'] == DATA_PERIOD) & (RUNS_INFO_DF['condition'] == DETECTOR_CONDITION), 'run_number'].values

# Selection
print(f"\nSelected {len(runs_to_analyze)} runs for merge:")
print(runs_to_analyze)


Selected 56 runs for merge:
[15625 15626 15627 15632 15633 15634 15635 15636 15637 15639 15640 15642
 15643 15644 15645 15647 15648 15649 15650 15655 15656 15657 15658 15659
 15669 15670 15671 15672 15673 15675 15676 15681 15682 15687 15688 15689
 15693 15694 15695 15696 15697 15698 15699 15700 15701 15709 15724 15729
 15730 15731 15732 15733 15734 15735 15736 15737]


In [10]:
total_corr_time = 0
# total_ok = 0
# total_lost = 0
total_processed_events = 0
all_processed_df = []

for run_id in runs_to_analyze:

    print(f"--- Merging Run {run_id} ---")
    if run_id not in SUMMARY_DF['Run_ID'].values:
        print(f"  → Run {run_id} not found in summary file. Skipping...")
        continue

    # --- Run Information --- #
    # Extract run information from the summary dataframe
    run_duration = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'Duration'].values[0]
    run_OK   = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'OK'].values[0]
    run_LOST = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'LOST'].values[0]
    # Calculate DAQ efficiency and corrected time
    DAQe_CV, DAQe_error = crudo.ff.efficiency(run_OK, run_LOST)
    run_corr_time    = run_duration * DAQe_CV
    total_corr_time += run_corr_time
    # Accumulate processed events
    processed_events = SUMMARY_DF.loc[SUMMARY_DF['Run_ID'] == run_id, 'Clean_Events'].values[0]
    total_processed_events += processed_events

    # Add run_number to dataframe for the global_event_id
    run_file = os.path.join(PROC_DATA_DIR, f'processed_run_{run_id}_{FILE_TAG}_events.h5')
    run_df = pd.read_hdf(run_file, key='Events')
    run_df['run_number'] = run_id
    all_processed_df.append(run_df)

# --- Print Summary --- #
print(f"\nFor period {DATA_PERIOD} with condition '{DETECTOR_CONDITION}':\n  Corrected Time = {total_corr_time:.4f} s")
# Concatenate all dataframes
MERGED_DF = pd.concat(all_processed_df, ignore_index=True)
print(f"Dataframe merged successfully:\n  Total processed events: {total_processed_events}")

--- Merging Run 15625 ---
--- Merging Run 15626 ---
--- Merging Run 15627 ---
--- Merging Run 15632 ---
--- Merging Run 15633 ---
--- Merging Run 15634 ---
--- Merging Run 15635 ---
--- Merging Run 15636 ---
--- Merging Run 15637 ---
--- Merging Run 15639 ---
--- Merging Run 15640 ---
--- Merging Run 15642 ---
--- Merging Run 15643 ---
--- Merging Run 15644 ---
--- Merging Run 15645 ---
--- Merging Run 15647 ---
--- Merging Run 15648 ---
--- Merging Run 15649 ---
--- Merging Run 15650 ---
--- Merging Run 15655 ---
--- Merging Run 15656 ---
--- Merging Run 15657 ---
--- Merging Run 15658 ---
--- Merging Run 15659 ---
--- Merging Run 15669 ---
--- Merging Run 15670 ---
--- Merging Run 15671 ---
--- Merging Run 15672 ---
--- Merging Run 15673 ---
--- Merging Run 15675 ---
--- Merging Run 15676 ---
--- Merging Run 15681 ---
--- Merging Run 15682 ---
--- Merging Run 15687 ---
--- Merging Run 15688 ---
--- Merging Run 15689 ---
--- Merging Run 15693 ---
--- Merging Run 15694 ---
--- Merging 

### MC

In [5]:
mc_files_to_merge = sorted(glob.glob(os.path.join(PROC_MC_DIR, f"*{FILE_TAG}*{DATE}*.h5")))
mc_files_to_merge

['/lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/mc/processed_mc_calibration_lpr_Tl208_19052026.h5']

In [6]:
all_processed_df = []

for file in mc_files_to_merge:
    print(f"--- Merging File: {os.path.basename(file)} ---")
    dataframe = pd.read_hdf(file, key='Events')
    all_processed_df.append(dataframe)

MERGED_DF = pd.concat(all_processed_df, ignore_index=True)

--- Merging File: processed_mc_calibration_lpr_Tl208_19052026.h5 ---


### Compute Global Event ID

In [7]:
if TYPE == 'data': COMP_COLS = ['event', 'run_number']
elif TYPE == 'mc': COMP_COLS = ['event', 'isotope', 'volume']

# An original event is defined as a row in dataframe where at least one of the columns 
# ('event', 'run_number') differs from the corresponding row below it (using `shift`).
event_OG = (MERGED_DF[COMP_COLS] != MERGED_DF[COMP_COLS].shift())

# If any column in event_OG is True, it means the row corresponds to the start of a new original event block.
new_event_block = event_OG.any(axis=1)

# Use `cumsum()` on the boolean mask to create a unique identifier for each contiguous block of hits 
# that belong to the same original event.
unique_block_id = new_event_block.cumsum()

# Assign a unique global event ID to each block of original events.
# The `factorize` function generates a unique integer code for each unique block ID.
MERGED_DF['global_event'] = pd.factorize(unique_block_id)[0]
print(f"{MERGED_DF['global_event'].nunique()} unique global events identified.")

131001 unique global events identified.


In [8]:
MERGED_DF

,event,nS1,nS2,isotope,volume,double_e,old_n_hits,time,S1e,S1e_corr,...,Y_bary,Z_bary,X_min,X_max,Y_min,Y_max,Z_min,Z_max,R_max,global_event
0,0,1,1,Tl208,PORT_1a,True,550.0,1.403882e+06,468.0,676.407688,...,240.114667,416.424938,-173.625,-50.225,168.175,291.575,364.979763,461.597477,318.023004,0
1,1,1,1,Tl208,PORT_1a,False,299.0,3.768631e+06,249.0,333.076721,...,303.919723,554.365940,288.875,412.275,245.425,368.825,521.529819,586.973612,509.200880,1
2,2,1,1,Tl208,PORT_1a,False,201.0,4.564927e+06,179.0,342.969833,...,-199.325629,27.654857,335.525,458.925,-263.725,-140.325,15.933211,34.756915,508.106506,2
3,3,1,2,Tl208,PORT_1a,False,363.0,6.753959e+06,335.0,507.693007,...,34.354366,692.056250,-19.125,243.225,-63.075,461.625,353.869917,755.658782,514.716957,3
4,3,1,2,Tl208,PORT_1a,False,363.0,6.753959e+06,335.0,411.642709,...,34.354366,692.056250,-19.125,243.225,-63.075,461.625,353.869917,755.658782,514.716957,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204571,4999022,1,2,Tl208,PORT_1a,False,257.0,9.998045e+12,271.0,372.852861,...,425.478144,482.111404,57.625,227.675,369.325,477.175,459.912994,529.355687,491.989849,130997
204572,4999023,1,1,Tl208,PORT_1a,False,171.0,9.998047e+12,208.0,350.395322,...,471.225306,210.320422,-81.325,10.975,430.525,492.725,196.425922,223.083143,499.391311,130998
204573,4999024,1,1,Tl208,PORT_1a,True,762.0,9.998049e+12,684.0,1069.626724,...,321.803227,286.417988,196.575,335.525,245.425,399.925,225.733435,336.126376,500.128125,130999
204574,4999025,1,2,Tl208,PORT_1a,False,293.0,9.998052e+12,222.0,391.254722,...,-32.100375,253.967627,-297.025,-142.525,-94.175,60.325,137.866929,284.873045,298.795041,131000


# Tagging Events

### By Particle

In [13]:
particle_tagged_MERGED_DF = crudo.dm.tag_particles( MERGED_DF
                                                  , size_threshold=SIZE_THRESHOLD
                                                  , s1_energy_threshold=S1_ENERGY_THRESHOLD
                                                  , event_column='global_event' )
particle_tagged_MERGED_DF

,event,nS1,nS2,old_n_hits,time,S1e,S1e_corr,S1w,S1h,S1t,...,X_min,X_max,Y_min,Y_max,Z_min,Z_max,R_max,run_number,global_event,particle
0,1058,1,1,8233,1.754042e+09,1348.162720,1338.559437,800.0,237.075485,49775.0,...,-482.625,490.025,-479.925,492.725,1165.845054,1203.139853,494.279785,15625,0,alpha
1,2206,1,1,830,1.754042e+09,89.690819,106.070783,350.0,13.100248,484750.0,...,-389.325,-220.275,-155.875,-16.425,780.906971,818.263077,400.435964,15625,1,electron
2,3102,0,1,869,1.754042e+09,NaN,NaN,NaN,NaN,NaN,...,-482.625,351.075,-171.425,384.375,136.444451,145.976535,572.328604,15625,2,electron
3,5461,1,1,1380,1.754042e+09,902.388672,1378.194141,800.0,155.612564,1008000.0,...,-404.875,412.275,-402.675,461.625,338.059516,357.730589,507.585595,15625,3,alpha
4,6126,1,1,773,1.754042e+09,1024.439575,1390.200754,625.0,171.154953,782325.0,...,-482.625,-297.025,-294.825,-48.025,532.266397,555.681839,488.325226,15625,4,alpha
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1026376,1789053,1,1,7630,1.759302e+09,1407.644653,1397.452512,650.0,251.366013,49450.0,...,-482.625,474.475,-479.925,492.725,1163.695637,1204.315713,506.672815,15737,939941,alpha
1026377,1789270,1,1,8047,1.759302e+09,1378.959717,1368.765830,900.0,233.034393,49025.0,...,-482.625,490.025,-479.925,492.725,1164.829003,1209.152468,500.123641,15737,939942,alpha
1026378,1790145,0,1,7346,1.759302e+09,NaN,NaN,NaN,NaN,NaN,...,-482.625,490.025,-479.925,492.725,19.056382,63.463104,498.160472,15737,939943,alpha
1026379,1790215,1,1,9115,1.759302e+09,1307.270386,1474.721141,675.0,222.353699,377450.0,...,-482.625,490.025,-479.925,492.725,881.601513,915.386142,496.083225,15737,939944,alpha


### By Detector Region

In [9]:
region_tagged_MERGED_DF = crudo.dm.tag_event_by_detector_region( particle_tagged_MERGED_DF if TYPE == 'data' else MERGED_DF
                                                               , z_cut_low=Z_LOW
                                                               , z_cut_high=Z_UP
                                                               , r_cut_high=R_UP
                                                               , event_column='global_event' )
region_tagged_MERGED_DF

,event,nS1,nS2,isotope,volume,double_e,old_n_hits,time,S1e,S1e_corr,...,Z_bary,X_min,X_max,Y_min,Y_max,Z_min,Z_max,R_max,global_event,region
0,0,1,1,Tl208,PORT_1a,True,550.0,1.403882e+06,468.0,676.407688,...,416.424938,-173.625,-50.225,168.175,291.575,364.979763,461.597477,318.023004,0,fiducial
1,1,1,1,Tl208,PORT_1a,False,299.0,3.768631e+06,249.0,333.076721,...,554.365940,288.875,412.275,245.425,368.825,521.529819,586.973612,509.200880,1,tube
2,2,1,1,Tl208,PORT_1a,False,201.0,4.564927e+06,179.0,342.969833,...,27.654857,335.525,458.925,-263.725,-140.325,15.933211,34.756915,508.106506,2,anode
3,3,1,2,Tl208,PORT_1a,False,363.0,6.753959e+06,335.0,507.693007,...,692.056250,-19.125,243.225,-63.075,461.625,353.869917,755.658782,514.716957,3,tube
4,3,1,2,Tl208,PORT_1a,False,363.0,6.753959e+06,335.0,411.642709,...,692.056250,-19.125,243.225,-63.075,461.625,353.869917,755.658782,514.716957,3,tube
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204571,4999022,1,2,Tl208,PORT_1a,False,257.0,9.998045e+12,271.0,372.852861,...,482.111404,57.625,227.675,369.325,477.175,459.912994,529.355687,491.989849,130997,tube
204572,4999023,1,1,Tl208,PORT_1a,False,171.0,9.998047e+12,208.0,350.395322,...,210.320422,-81.325,10.975,430.525,492.725,196.425922,223.083143,499.391311,130998,tube
204573,4999024,1,1,Tl208,PORT_1a,True,762.0,9.998049e+12,684.0,1069.626724,...,286.417988,196.575,335.525,245.425,399.925,225.733435,336.126376,500.128125,130999,tube
204574,4999025,1,2,Tl208,PORT_1a,False,293.0,9.998052e+12,222.0,391.254722,...,253.967627,-297.025,-142.525,-94.175,60.325,137.866929,284.873045,298.795041,131000,fiducial


# Output

In [10]:
FINAL_DF = region_tagged_MERGED_DF[FINAL_COLS].copy()
FINAL_DF

,event,global_event,time,isotope,volume,double_e,region,nS1,nS2,n_cluster,...,S1e,S1e_corr,S1w,S1h,S1t,S2e,S2w,S2h,S2t,S2q
0,0,0,1.403882e+06,Tl208,PORT_1a,True,fiducial,1,1,1,...,468.0,676.407688,550.0,113.0,10000.0,412496.0,130.875,10751.0,513493.218750,14370.492188
1,1,1,3.768631e+06,Tl208,PORT_1a,False,tube,1,1,1,...,249.0,333.076721,525.0,55.0,10000.0,159583.0,98.425,5255.0,666483.625000,5489.101074
2,2,2,4.564927e+06,Tl208,PORT_1a,False,anode,1,1,1,...,179.0,342.969833,575.0,35.0,10000.0,198132.0,29.575,19978.0,46490.390625,7220.463379
3,3,3,6.753959e+06,Tl208,PORT_1a,False,tube,1,2,3,...,335.0,507.693007,550.0,71.0,10025.0,19042.0,29.450,1849.0,425483.062500,670.840576
4,3,3,6.753959e+06,Tl208,PORT_1a,False,tube,1,2,3,...,335.0,411.642709,550.0,71.0,10025.0,236672.0,131.225,7883.0,848489.625000,7804.173340
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204571,4999022,130997,9.998045e+12,Tl208,PORT_1a,False,tube,1,2,3,...,271.0,372.852861,575.0,60.0,10000.0,40656.0,50.375,2576.0,609482.187500,1242.203003
204572,4999023,130998,9.998047e+12,Tl208,PORT_1a,False,tube,1,1,1,...,208.0,350.395322,425.0,39.0,10000.0,132060.0,44.225,6416.0,243489.921875,5364.290039
204573,4999024,130999,9.998049e+12,Tl208,PORT_1a,True,tube,1,1,1,...,684.0,1069.626724,525.0,155.0,10000.0,576357.0,145.075,14200.0,369487.156250,22211.998047
204574,4999025,131000,9.998052e+12,Tl208,PORT_1a,False,fiducial,1,2,2,...,222.0,391.254722,475.0,51.0,10000.0,12179.0,17.725,1787.0,171480.187500,233.101456


In [11]:
# H5 output filename
merged_filename = 'merged_tagged_'
if TYPE == 'data': merged_filename += 'runs_' + DETECTOR_CONDITION + '_'
elif TYPE == 'mc': merged_filename += 'mc_'
merged_filename += FILE_TAG + '.h5'
    
merged_path = os.path.join(OUTPUT_DIR, merged_filename)
print(f"\nSaving merged dataframe to: {merged_path}")


Saving merged dataframe to: /lustre/ific.uv.es/prj/gl/neutrinos/users/ccortesp/NEXT-100/Backgrounds/h5/merged_tagged_mc_calibration_lpr.h5


In [12]:
FINAL_DF.to_hdf(merged_path, key='Events', mode='w', format='table')
print('Done!')

Done!
